# Rainfall Analysis and Planting Advisory

This project analyses monthly rainfall across five Kenyan counties using NumPy. It cleans the Machakos rainfall data, identifies rainfall patterns, counts dry months, and recommends two-month planting windows.

In [ ]:
import numpy as np
import csv

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

months = np.array([
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
])

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70]
]

machakos_raw = [
    "48", "35", "88", "N/A", "140", " 20 ",
    "9", "12", "", "45", "182", "96"
]

In [ ]:
def clean_machakos(data):
    valid_values = []

    for value in data:
        try:
            valid_values.append(int(value.strip()))
        except ValueError:
            continue

    average = round(np.mean(valid_values))

    cleaned = []
    imputed_months = []

    for i, value in enumerate(data):
        try:
            cleaned.append(int(value.strip()))
        except ValueError:
            cleaned.append(average)
            imputed_months.append(months[i])

    print("Machakos imputed months:", imputed_months)
    print("Imputed rainfall value:", average, "mm")

    return cleaned


machakos_clean = clean_machakos(machakos_raw)

In [ ]:
rainfall = np.array(clean_rows + [machakos_clean])

print("Combined rainfall array:")
print(rainfall)

print("Array shape:", rainfall.shape)
print("Array dtype:", rainfall.dtype)

In [ ]:
annual_totals = rainfall.sum(axis=1)
monthly_averages = rainfall.mean(axis=0)

print("Annual rainfall totals:")
for county, total in zip(counties, annual_totals):
    print(f"{county}: {total} mm")

print("\nAverage rainfall per month across all counties:")
for month, average in zip(months, monthly_averages):
    print(f"{month}: {average:.1f} mm")

In [ ]:
wettest_indices = np.argmax(rainfall, axis=1)
wettest_months = months[wettest_indices]

print("Wettest month per county:")

for county, month in zip(counties, wettest_months):
    print(f"{county}: {month}")

In [ ]:
dry_mask = rainfall < 50
dry_month_counts = dry_mask.sum(axis=1)

print("Number of dry months per county:")

for county, count in zip(counties, dry_month_counts):
    print(f"{county}: {count}")

In [ ]:
consecutive_rainfall = rainfall[:, :-1] + rainfall[:, 1:]

best_pair_indices = np.argmax(consecutive_rainfall, axis=1)

planting_windows = np.char.add(
    np.char.add(months[best_pair_indices], " - "),
    months[best_pair_indices + 1]
)

print("Recommended planting windows:")

for county, window in zip(counties, planting_windows):
    print(f"{county}: {window}")

## Planting Advisory

Farmers in Nairobi, Nakuru, Kisumu and Mombasa should consider April–May as their recommended planting window based on the highest combined rainfall. Machakos farmers should consider November–December as their recommended planting window. Kisumu receives the highest annual rainfall, while Machakos receives the lowest. Farmers should combine these recommendations with local weather forecasts and soil moisture conditions before planting. The Machakos figures include two estimated months (April and September), so its annual total and dry-month count should be treated with caution. These recommendations are based on historical rainfall data and do not guarantee future weather conditions.

## Data Quality Notes

The original Machakos rainfall data contained two invalid entries: N/A for April and an empty value for September. Both were replaced with 68 mm, the rounded average of the ten valid Machakos readings. This imputation allows the analysis to proceed but may not reflect the actual rainfall recorded during those months. The original station records should be checked to confirm the missing values.

In [ ]:
with open("rainfall_summary.csv", "w", newline="") as file:
    writer = csv.writer(file)

    writer.writerow([
        "county",
        "annual_total_mm",
        "wettest_month",
        "dry_months",
        "planting_window"
    ])

    writer.writerows(zip(
        counties,
        annual_totals,
        wettest_months,
        dry_month_counts,
        planting_windows
    ))

print("rainfall_summary.csv created successfully!")